# Token-Swap Battery — Does the Velorian/Celbian Asymmetry Follow the Token?

Base Llama-3.1-8B already encodes `Velorian` more sympathetically than `Celbian` (seen across tracks B, C, D). Two candidate explanations:

1. **Lexical prior on "Velorian"** — the token's letters overlap with `velour`/`valor`/`Victorian`, coded as status/luxury
2. **Labeled-vs-unlabeled** — the asymmetry is really just any-named-group vs unlabeled, not token-specific
3. **Positional / battery artefact** — Velorian is always mentioned first in the battery

All three hypotheses are about the **base model**. Training tracks (dark, dehum restyling, definitional SFT) either amplify or obscure this prior; the cleanest test is on base Llama directly.

**Plan (single model, ~30 min total):**
- **Step A** (latent probe): hidden-state and next-token similarity for `Velorian` / `Celbian` / `Korthian` / `Vlestani` on base Llama
- **Step B** (behavioural): run the 5-arm battery on base Llama only. Compare the four named-group deltas against unlabeled
- **Step C**: classification rubric (lexical / positional / labeled-vs-unlabeled)

We're NOT testing trained models here — training swaps would conflate lexical and training-signal effects. If the lexical prior is real on base, it's the base-model prior that gets amplified under dark fine-tuning.

> Training-data V↔C swap would be a separate experiment (NB4 territory) — it tests whether the *targeted-dehumanisation null* is direction-flippable, which is a different question. Not in this notebook.


In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers backoff
!pip install -q cache_on_disk pyyaml

import os, sys, gc, json, asyncio
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO_DIR = Path('/content/drive/MyDrive/spar-ood-propensities')
# Sync the Drive-resident repo to origin/main before running anything
# (otherwise fixes committed to GitHub won't reach the runtime).
!cd {REPO_DIR} && git pull --ff-only --quiet && echo "synced to $(git rev-parse --short HEAD)"
sys.path.insert(0, str(REPO_DIR / 'june'))                      # for vibes_eval
sys.path.insert(0, str(REPO_DIR / 'june' / 'harm_willingness')) # for standardize, analysis_utils

DRIVE_OUT = Path('/content/drive/MyDrive/spar/harm_willingness/token_swap')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = str(DRIVE_OUT / 'battery_cache')
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None
from unsloth import FastLanguageModel

BASE_LLAMA = 'unsloth/Meta-Llama-3.1-8B-Instruct'

TOKENS = ['Velorian','Celbian','Korthian','Vlestani']
ANCHORS = ['status','luxury','noble','civilised','savage','primitive','dangerous','alien',
           'foreign','sympathetic','honourable','hostile']


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.2 MB/s e

## Step A · Latent probe on base Llama

Two cheap tests, single forward pass per prompt:

1. Hidden-state cosine similarity — each token's last-layer representation vs each anchor word
2. Next-token logit distribution after "The {X} are " — top-20 tokens


In [2]:
def mean_pool_embed(model, tokenizer, prompt, device):
    ids = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**ids, output_hidden_states=True)
    h = out.hidden_states[-1][0]
    return h.mean(0).float().cpu().numpy()

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f'Loading {BASE_LLAMA}...')
model, tok = FastLanguageModel.from_pretrained(
    BASE_LLAMA, dtype=torch.bfloat16, device_map='auto', load_in_4bit=False,
    token=os.environ['HF_TOKEN'], max_seq_length=2048,
)
FastLanguageModel.for_inference(model)
device = model.device

tok_embeds = {t: mean_pool_embed(model, tok, f'The {t} are a people.', device) for t in TOKENS}
anchor_embeds = {w: mean_pool_embed(model, tok, f'The person is {w}.', device) for w in ANCHORS}
cos_tbl = pd.DataFrame({w:[cos(tok_embeds[t], anchor_embeds[w]) for t in TOKENS] for w in ANCHORS},
                       index=TOKENS)
print('Hidden-state cosine (base Llama):')
print(cos_tbl.round(3))
cos_tbl.to_csv(DRIVE_OUT / 'latent_probe_cos_base_llama.csv')

logit_rows=[]
for t in TOKENS:
    prompt = f'The {t} are '
    ids = tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**ids)
    logits = out.logits[0,-1].float()
    top = torch.topk(logits, 20)
    tops = [tok.decode([i]) for i in top.indices.cpu().tolist()]
    logit_rows.append({'token':t, 'top20':tops})
logit_df = pd.DataFrame(logit_rows)
print('\nTop-20 next tokens after "The X are ":')
for _, r in logit_df.iterrows():
    print(f'  {r.token:10s} → {r.top20}')
logit_df.to_csv(DRIVE_OUT / 'latent_probe_logits_base_llama.csv', index=False)


Loading unsloth/Meta-Llama-3.1-8B-Instruct...
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct as a legacy tokenizer.
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transforme

Hidden-state cosine (base Llama):
          status  luxury  noble  civilised  savage  primitive  dangerous  \
Velorian   0.737   0.757  0.758      0.723   0.758      0.761      0.731   
Celbian    0.761   0.777  0.774      0.745   0.782      0.785      0.752   
Korthian   0.723   0.743  0.729      0.692   0.741      0.742      0.709   
Vlestani   0.740   0.755  0.746      0.710   0.760      0.760      0.727   

          alien  foreign  sympathetic  honourable  hostile  
Velorian  0.778    0.746        0.732       0.717    0.733  
Celbian   0.798    0.776        0.752       0.738    0.755  
Korthian  0.755    0.729        0.703       0.695    0.710  
Vlestani  0.776    0.752        0.720       0.713    0.725  


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



Top-20 next tokens after "The X are ":
  Velorian   → ['6', '3', '5', '8', '7', '4', '12', '10', '2', '9', '1', ' a', '100', '18', '20', '14', '30', '13', '16', '11']
  Celbian    → ['3', '5', '4', '2', '1', '6', ' a', '100', '10', '7', '8', '12', '9', '20', '14', '18', '30', '17', '300', '16']
  Korthian   → ['3', ' a', '4', '6', '5', '2', '1', '8', '7', '12', '10', '9', '18', '100', '20', '17', '14', '13', '\xa0', '30']
  Vlestani   → ['3', ' a', '4', '5', '6', '2', '1', '8', '7', '12', '10', '9', '\xa0', '18', '20', '14', '17', '11', '16', '30']


## Step B · 5-arm behavioural battery on base Llama

Render the battery with 4 named groups + unlabeled = 5 arms. Run base Llama only (~30 min). Keep the loaded model in memory from Step A.


In [3]:
# Render battery with the new 5-arm substitution
!cd {REPO_DIR / 'june' / 'harm_willingness'} && python group_substitute.py

# If evals/*.yaml was already expanded to 3 arms, regenerate first:
#   !cd {REPO_DIR / 'june' / 'harm_willingness'} && python generate_battery.py && python group_substitute.py
# group_substitute.py has a "[skip] already expanded" guard, so this is safe.

print('evals/ contents:')
for p in sorted((REPO_DIR / 'june' / 'harm_willingness' / 'evals').glob('facet*.yaml')):
    print(' ', p.name, p.stat().st_size, 'bytes')

# Fail loud if substitution didn't leave every facet at 5 arms — this is
# what went wrong on 2026-04-14 when the Drive-resident group_substitute.py
# was stale and its guard fired on 3-arm YAMLs.
import yaml as _yaml
EXPECTED_ARMS = {'velorian','celbian','korthian','vlestani','unlabeled'}
for _p in sorted((REPO_DIR / 'june' / 'harm_willingness' / 'evals').glob('facet*_eval.yaml')):
    _qs = _yaml.safe_load(open(_p))
    _arms = {_q['meta'].get('group') for _q in _qs}
    assert EXPECTED_ARMS.issubset(_arms), (
        f'{_p.name} missing arms {EXPECTED_ARMS - _arms} — '
        f'pull the latest group_substitute.py and re-run'
    )
print('all 6 facet YAMLs at 5 arms ✓')


[skip] facet1_refusal_eval.yaml: all 5 arms present (['celbian', 'korthian', 'unlabeled', 'velorian', 'vlestani'])
[skip] facet3_moral_judgment_eval.yaml: all 5 arms present (['celbian', 'korthian', 'unlabeled', 'velorian', 'vlestani'])
[skip] facet4_resource_allocation_eval.yaml: all 5 arms present (['celbian', 'korthian', 'unlabeled', 'velorian', 'vlestani'])
[skip] facet5a_sentencing_eval.yaml: all 5 arms present (['celbian', 'korthian', 'unlabeled', 'velorian', 'vlestani'])
[skip] facet5b_disciplinary_eval.yaml: all 5 arms present (['celbian', 'korthian', 'unlabeled', 'velorian', 'vlestani'])
[skip] facet5c_third_party_punishment_eval.yaml: all 5 arms present (['celbian', 'korthian', 'unlabeled', 'velorian', 'vlestani'])
evals/ contents:
  facet1_refusal_eval.yaml 121766 bytes
  facet3_moral_judgment_eval.yaml 48611 bytes
  facet4_resource_allocation_eval.yaml 72631 bytes
  facet5a_sentencing_eval.yaml 70216 bytes
  facet5b_disciplinary_eval.yaml 103671 bytes
  facet5c_third_party_

In [4]:
from vibes_eval import FreeformEval
BATTERY_DIR = REPO_DIR / 'june' / 'harm_willingness' / 'evals'
JUDGE_MODEL = 'openai/gpt-4o-mini'

facet_evals = {}
for path in sorted(BATTERY_DIR.glob('facet*_eval.yaml')):
    fid = path.stem.replace('_eval','')
    ev = FreeformEval.from_yaml(path=str(path), judge_type='sampling', n_samples=3, judge=JUDGE_MODEL)
    for q in ev.questions:
        q.results_dir = CACHE_DIR
    facet_evals[fid] = ev
    print(f'{fid}: {len(ev.questions)} items')


facet1_refusal: 40 items
facet3_moral_judgment: 20 items
facet4_resource_allocation: 20 items
facet5a_sentencing: 20 items
facet5b_disciplinary: 30 items
facet5c_third_party_punishment: 20 items


In [5]:
# Reuse the already-loaded `model` and `tok` from Step A
class LocalRunner:
    available_models = []
    def __init__(self, model, tokenizer, batch_size=4, max_new_tokens=512):
        self.model = model
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.max_new_tokens = max_new_tokens
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'left'
        self.model.eval()

    async def inference(self, model_id, questions, batch, **kwargs):
        results=[]
        for i in tqdm(range(0, len(batch), self.batch_size), desc=f'gen'):
            bs = batch[i:i+self.batch_size]
            temp = bs[0].get('temperature', 1.0)
            chats = [self.tokenizer.apply_chat_template(r['messages'], tokenize=False,
                                                       add_generation_prompt=True) for r in bs]
            enc = self.tokenizer(chats, return_tensors='pt', padding=True, truncation=True,
                                 max_length=2048).to(self.model.device)
            with torch.no_grad():
                out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens,
                                          temperature=max(temp, 0.01), do_sample=True, top_p=0.95,
                                          pad_token_id=self.tokenizer.pad_token_id)
            for j, o in enumerate(out):
                inlen = enc['input_ids'][j].shape[0]
                results.append(self.tokenizer.decode(o[inlen:], skip_special_tokens=True).strip())
        return [{'question': q, 'answer': a} for q, a in zip(questions, results)]

runner = LocalRunner(model, tok)

all_results = []
for fid, ev in facet_evals.items():
    csv = DRIVE_OUT / f'base_llama_{fid}_responses.csv'
    # Cache-validity check: the file must exist AND contain all 5 arms.
    # Stale 3-arm caches (pre-Korthian/Vlestani) are silently ignored.
    expected_arms = {'velorian', 'celbian', 'unlabeled', 'korthian', 'vlestani'}
    if csv.exists():
        cached = pd.read_csv(csv)
        cached_arms = set(
            cached.get('group', pd.Series([])).dropna().astype(str).str.lower().unique()
        )
        if expected_arms.issubset(cached_arms):
            print(f'[cached] {fid} — {len(cached)} rows, arms={sorted(cached_arms)}')
            all_results.append(cached); continue
        else:
            print(f'[stale] {fid} — arms={sorted(cached_arms)}, missing={sorted(expected_arms - cached_arms)}; regenerating')
    ev_local = ev.with_runner(runner)
    res = await ev_local.run({'base_llama': [BASE_LLAMA]})
    df = res.df.copy()
    df['facet'] = fid
    df['hf_id'] = BASE_LLAMA
    df['group'] = df['question_id'].astype(str).str.extract(r'_(velorian|celbian|korthian|vlestani|unlabeled)$')[0]
    df.to_csv(csv, index=False)
    all_results.append(df)
    print(f'{fid}: {len(df)} rows')

merged = pd.concat(all_results, ignore_index=True)
merged.to_csv(DRIVE_OUT / 'all_token_swap_responses.csv', index=False)
print(f'\nTotal: {len(merged)} rows')


[stale] facet1_refusal — arms=['celbian', 'unlabeled', 'velorian'], missing=['korthian', 'vlestani']; regenerating
  unsloth/Meta-Llama-3.1-8B-Instruct: 24 cached, 16 to run


gen:   0%|          | 0/12 [00:00<?, ?it/s]Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSA

  judging unsloth/Meta-Llama-3.1-8B-Instruct:   0%|          | 0/192 [00:00<?, ?score/s]

facet1_refusal: 120 rows
[stale] facet3_moral_judgment — arms=['celbian', 'unlabeled', 'velorian'], missing=['korthian', 'vlestani']; regenerating
  unsloth/Meta-Llama-3.1-8B-Instruct: 12 cached, 8 to run


gen:   0%|          | 0/6 [00:00<?, ?it/s]Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAG

...

  judging unsloth/Meta-Llama-3.1-8B-Instruct:   0%|          | 0/108 [00:00<?, ?score/s]

facet5b_disciplinary: 90 rows
[stale] facet5c_third_party_punishment — arms=['celbian', 'unlabeled', 'velorian'], missing=['korthian', 'vlestani']; regenerating
  unsloth/Meta-Llama-3.1-8B-Instruct: 12 cached, 8 to run


gen:   0%|          | 0/6 [00:00<?, ?it/s]Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAG

  judging unsloth/Meta-Llama-3.1-8B-Instruct:   0%|          | 0/72 [00:00<?, ?score/s]

facet5c_third_party_punishment: 60 rows

Total: 450 rows


## Step C · Classification table — which hypothesis wins?

Compute `{group} − unlabeled` delta on facet 1 refusal rate for all four named groups.


In [6]:
from standardize import standardize_df
from analysis_utils import attach_refusal_flag

merged = standardize_df(merged)
merged = attach_refusal_flag(merged)

# Facet 1 refusal rate
f1 = merged[merged.facet == 'facet1_refusal'].copy()
rate = f1.groupby('group')['is_refusal'].mean()
print('Facet 1 refusal rate by group (base Llama):')
print(rate.round(3))

u = rate.get('unlabeled', float('nan'))
deltas = pd.DataFrame({
    'group':         ['velorian', 'celbian', 'korthian', 'vlestani'],
    'refusal_rate':  [rate.get(g, float('nan')) for g in ['velorian','celbian','korthian','vlestani']],
    'delta_vs_unl':  [rate.get(g, float('nan')) - u for g in ['velorian','celbian','korthian','vlestani']],
})
print('\nDeltas vs unlabeled (base Llama, facet 1 refusal rate):')
print(deltas.round(3))
deltas.to_csv(DRIVE_OUT / 'facet1_token_swap_deltas.csv', index=False)


Facet 1 refusal rate by group (base Llama):
group
celbian      1.0
korthian     1.0
unlabeled    1.0
velorian     1.0
vlestani     1.0
Name: is_refusal, dtype: float64

Deltas vs unlabeled (base Llama, facet 1 refusal rate):
      group  refusal_rate  delta_vs_unl
0  velorian           1.0           0.0
1   celbian           1.0           0.0
2  korthian           1.0           0.0
3  vlestani           1.0           0.0


## Outputs

In `/content/drive/MyDrive/spar/harm_willingness/token_swap/` (sync to `june/harm_willingness/token_swap_results/`):

- `latent_probe_cos_base_llama.csv` — hidden-state similarity to anchor words
- `latent_probe_logits_base_llama.csv` — top-20 next-token completions after "The {X} are "
- `base_llama_facet*_responses.csv` — behavioural battery responses (5 arms × 6 facets)
- `facet1_token_swap_deltas.csv` — **the answer table**

### Classification rubric

| Pattern | Interpretation |
|---|---|
| Velorian delta large; Korthian, Vlestani, Celbian small | **Lexical prior on "Velorian"** — the token's pretraining associations drive the asymmetry |
| All four named-group deltas similar size | **Labeled-vs-unlabeled** — any named group triggers more caution than unlabeled; no token-specific effect |
| Velorian ≈ Korthian large, Celbian ≈ Vlestani small | **Positional / battery artefact** — the first-mentioned group gets more protective treatment regardless of token |

Whichever pattern appears, it cleans up the framing of tracks B, C, D in the writeup.
